In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from statsmodels.stats.contingency_tables import mcnemar
import warnings
warnings.filterwarnings("ignore")

In [6]:
def mcnemar_test(y_true, y_pred_ours, y_pred_baseline, baseline_name):
    """
    McNemar's test compares two classifiers on the same test set.
    Tests whether our model and baseline make different errors.
    H0: Both models have the same error rate.
    p < 0.05 means our improvement is statistically significant.
    """
    # Build contingency table
    # a = both correct
    # b = ours correct, baseline wrong  
    # c = ours wrong, baseline correct
    # d = both wrong
    a = np.sum((y_pred_ours == y_true) & (y_pred_baseline == y_true))
    b = np.sum((y_pred_ours == y_true) & (y_pred_baseline != y_true))
    c = np.sum((y_pred_ours != y_true) & (y_pred_baseline == y_true))
    d = np.sum((y_pred_ours != y_true) & (y_pred_baseline != y_true))

    table = [[a, b], [c, d]]
    result = mcnemar(table, exact=False, correction=True)

    significant = "YES ***" if result.pvalue < 0.05 else "NO"
    print(f"  vs {baseline_name:<22} "
          f"b={b:>4}  c={c:>4}  "
          f"chi2={result.statistic:>7.4f}  "
          f"p={result.pvalue:>8.5f}  "
          f"Significant: {significant}")
    return {
        "baseline": baseline_name,
        "b": b, "c": c,
        "chi2": result.statistic,
        "p_value": result.pvalue,
        "significant": result.pvalue < 0.05
    }


def run_significance_tests(X, y, dataset_name):
    imputer = SimpleImputer(strategy="mean")
    X_imp = imputer.fit_transform(X)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    rf  = RandomForestClassifier(n_estimators=200, random_state=42)
    svm = Pipeline([("scaler", StandardScaler()),
                    ("clf", SVC(kernel="rbf", probability=True, random_state=42))])
    lr  = Pipeline([("scaler", StandardScaler()),
                    ("clf", LogisticRegression(max_iter=1000, random_state=42))])

    rf_preds, svm_preds, lr_preds, true_labels = [], [], [], []

    for train_idx, test_idx in skf.split(X_imp, y):
        X_train, X_test = X_imp[train_idx], X_imp[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        rf.fit(X_train, y_train)
        svm.fit(X_train, y_train)
        lr.fit(X_train, y_train)

        rf_preds.extend(rf.predict(X_test))
        svm_preds.extend(svm.predict(X_test))
        lr_preds.extend(lr.predict(X_test))
        true_labels.extend(y_test)

    rf_preds   = np.array(rf_preds)
    svm_preds  = np.array(svm_preds)
    lr_preds   = np.array(lr_preds)
    true_labels = np.array(true_labels)

    print(f"\n{'='*70}")
    print(f"  McNemar's Test: {dataset_name}")
    print(f"  H0: Random Forest and baseline have equal error rates")
    print(f"  p < 0.05 = statistically significant improvement")
    print(f"{'='*70}")
    print(f"  {'Comparison':<26} {'b':>5} {'c':>5} "
          f"{'chi2':>10} {'p-value':>10} {'Significant'}")
    print(f"  {'-'*66}")

    results = []
    results.append(mcnemar_test(true_labels, rf_preds, svm_preds, "SVM (RBF)"))
    results.append(mcnemar_test(true_labels, rf_preds, lr_preds,  "Logistic Regression"))

    return results

In [7]:
feature_cols = [
    "pitch", "spectral_centroid", "zcr",
    "jitter", "shimmer", "hnr",
    "mfcc_1","mfcc_2","mfcc_3","mfcc_4","mfcc_5","mfcc_6","mfcc_7",
    "mfcc_8","mfcc_9","mfcc_10","mfcc_11","mfcc_12","mfcc_13"
]
pk_cols = ["pitch", "hnr", "jitter", "shimmer", "nhr", "rpde", "dfa"]

df_pk = pd.read_csv("../data/parkinsons_tabular/parkinsons.csv").rename(columns={
    "MDVP:Fo(Hz)": "pitch", "HNR": "hnr",
    "MDVP:Jitter(Abs)": "jitter", "MDVP:Shimmer": "shimmer",
    "NHR": "nhr", "RPDE": "rpde", "DFA": "dfa"
})
df_resp   = pd.read_csv("../data/voice_features/voice_dataset_labeled_full.csv").dropna()
df_stress = pd.read_csv("../data/voice_features/stress_dataset.csv").dropna()
df_depr   = pd.read_csv("../data/voice_features/depression_dataset.csv").dropna()

res_pk     = run_significance_tests(df_pk[pk_cols],          df_pk["status"],    "Parkinson's Disease")
res_resp   = run_significance_tests(df_resp[feature_cols],   df_resp["label"],   "Respiratory Abnormality")
res_stress = run_significance_tests(df_stress[feature_cols], df_stress["label"], "Psychological Stress")
res_depr   = run_significance_tests(df_depr[feature_cols],   df_depr["label"],   "Depression Indicators")


  McNemar's Test: Parkinson's Disease
  H0: Random Forest and baseline have equal error rates
  p < 0.05 = statistically significant improvement
  Comparison                     b     c       chi2    p-value Significant
  ------------------------------------------------------------------
  vs SVM (RBF)              b=  11  c=   9  chi2= 0.0500  p= 0.82306  Significant: NO
  vs Logistic Regression    b=  18  c=   7  chi2= 4.0000  p= 0.04550  Significant: YES ***

  McNemar's Test: Respiratory Abnormality
  H0: Random Forest and baseline have equal error rates
  p < 0.05 = statistically significant improvement
  Comparison                     b     c       chi2    p-value Significant
  ------------------------------------------------------------------
  vs SVM (RBF)              b= 358  c= 357  chi2= 0.0000  p= 1.00000  Significant: NO
  vs Logistic Regression    b= 655  c= 367  chi2=80.5959  p= 0.00000  Significant: YES ***

  McNemar's Test: Psychological Stress
  H0: Random Forest an

In [8]:
import os
os.makedirs("../results", exist_ok=True)

all_stats = []
for name, res in [
    ("Parkinson's", res_pk),
    ("Respiratory", res_resp),
    ("Stress", res_stress),
    ("Depression", res_depr)
]:
    for r in res:
        r["disease"] = name
        all_stats.append(r)

df_stats = pd.DataFrame(all_stats)
df_stats.to_csv("../results/statistical_tests.csv", index=False)
print("\nSaved: results/statistical_tests.csv")
print("\nSummary: How many comparisons are statistically significant?")
print(f"  {df_stats['significant'].sum()} out of {len(df_stats)} comparisons (p < 0.05)")


Saved: results/statistical_tests.csv

Summary: How many comparisons are statistically significant?
  4 out of 8 comparisons (p < 0.05)
